#KITCHEN PROLINE - Notebook 02 - Devis et Fact Table

**Objectif :**
Ce notebook construit la dimension DimDevis ainsi que la table de faits centrale FactLignesDevis à partir des fichiers de devis quotidiens (format CSV) issus du système de gestion des devis.

**Sources :**
Fichiers CSV dans `/Volumes/kitchen-proline-data/raw/data-files/Données/Devis/`
KTP1_*.csv : En-têtes des devis
KTP2_*.csv : Détails des devis (lignes de produits)

**Tables créées (schéma sales) :**
DimDevis, FactLignesDevis

**Transformations appliquées :**
Consolidation de tous les fichiers KTP1 et KTP2, extraction du CodeProduit via substring(CodeArticle, 4), jointures avec les 11 dimensions pour récupérer les clés techniques, gestion des valeurs nulles (IdModele → 0, IdFacade → 1), calcul de la marge brute (prix_net_ht - prix_achat_ht)

**Dépendances :**
Ce notebook doit être exécuté APRÈS le notebook 01

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Configuration
BASE_PATH_DEVIS = "/Volumes/kitchen-proline-data/raw/data-files/Données/Devis/"
CATALOG = "kitchen-proline-data"
SCHEMA = "sales"

# Fonction pour sauvegarder une table
def save_table(df, table_name, mode="overwrite"):
    """
    Sauvegarde un DataFrame comme table dans Unity Catalog
    """
    full_table_name = f"`{CATALOG}`.`{SCHEMA}`.`{table_name}`"
    df.write.mode(mode).saveAsTable(full_table_name)
    print(f"✓ Table {full_table_name} créée avec succès ({df.count()} lignes)")
    return full_table_name

print("✓ Configuration chargée")

## Exploration des fichiers de devis

In [0]:
# Lister tous les fichiers CSV dans les sous-répertoires KTP1 et KTP2
print("=== FICHIERS KTP1 (En-têtes) ===")
ktp1_files = dbutils.fs.ls(BASE_PATH_DEVIS + "KTP1/")
print(f"Nombre de fichiers KTP1 : {len(ktp1_files)}")
for f in ktp1_files[:5]:
    print(f"  - {f.name} ({f.size} bytes)")

print("\n=== FICHIERS KTP2 (Détails) ===")
ktp2_files = dbutils.fs.ls(BASE_PATH_DEVIS + "KTP2/")
print(f"Nombre de fichiers KTP2 : {len(ktp2_files)}")
for f in ktp2_files[:5]:
    print(f"  - {f.name} ({f.size} bytes)")

In [0]:
# Charger un fichier KTP1 pour voir la structure
print("=== FICHIER KTP1 (En-têtes de devis) ===")

# Prendre le premier fichier KTP1 non vide
fichiers_non_vides = [f for f in ktp1_files if f.size > 100]
if fichiers_non_vides:
    fichier_ktp1 = fichiers_non_vides[0].path
    print(f"Fichier test : {fichier_ktp1.split('/')[-1]}\n")
    
    df_ktp1_sample = spark.read.csv(
        fichier_ktp1,
        sep=";",
        header=True,
        inferSchema=True
    )
    
    print("Schéma :")
    df_ktp1_sample.printSchema()
    print(f"\nNombre de lignes : {df_ktp1_sample.count()}")
    print("\nÉchantillon :")
    display(df_ktp1_sample.limit(5))
else:
    print("⚠️ Aucun fichier KTP1 non vide trouvé")

In [0]:
# Charger un fichier KTP2 pour voir la structure
print("=== FICHIER KTP2 (Détails de devis) ===")

# Prendre le premier fichier KTP2 non vide
fichiers_non_vides = [f for f in ktp2_files if f.size > 100]
if fichiers_non_vides:
    fichier_ktp2 = fichiers_non_vides[0].path
    print(f"Fichier test : {fichier_ktp2.split('/')[-1]}\n")
    
    df_ktp2_sample = spark.read.csv(
        fichier_ktp2,
        sep=";",
        header=True,
        inferSchema=True
    )
    
    print("Schéma :")
    df_ktp2_sample.printSchema()
    print(f"\nNombre de lignes : {df_ktp2_sample.count()}")
    print("\nÉchantillon :")
    display(df_ktp2_sample.limit(5))
else:
    print(" Aucun fichier KTP2 non vide trouvé")

## Création de DimDevis depuis KTP1
Consolidation de tous les fichiers KTP1 (en-têtes de devis)

In [0]:
# Charger tous les fichiers KTP1
print("🔄 Chargement de tous les fichiers KTP1...")

# Utiliser un wildcard pour charger tous les fichiers CSV dans KTP1/
df_ktp1_all = spark.read.csv(
    BASE_PATH_DEVIS + "KTP1/*.csv",
    sep=";",
    header=True,
    inferSchema=True
)

print(f"✓ Nombre total de lignes chargées : {df_ktp1_all.count()}")
print(f"✓ Nombre de devis uniques : {df_ktp1_all.select('NUM_DEVIS').distinct().count()}")

print("\nSchéma des données :")
df_ktp1_all.printSchema()

print("\nÉchantillon des statuts :")
df_ktp1_all.groupBy("STATUT").count().orderBy(F.desc("count")).show()

In [0]:
# Transformation et nettoyage des données pour DimDevis avec surrogate key
print(" Transformation des données...")

from pyspark.sql.window import Window

# Première sélection sans l'ID
df_temp = df_ktp1_all.select(
    F.col("NUM_DEVIS").alias("numero_devis"),
    F.col("NUM_PROJ").alias("numero_projet"),
    F.expr("try_to_date(DATE_DEVIS, 'dd/MM/yyyy')").alias("date_devis"),
    F.expr("try_to_timestamp(DMOD, 'dd/MM/yyyy HH:mm')").alias("date_modification"),
    F.when(F.col("DATE_RDV").isNotNull(), F.expr("try_to_date(DATE_RDV, 'dd/MM/yyyy')")).alias("date_rendez_vous"),
    F.when(F.col("DATE_VENTE").isNotNull(), F.expr("try_to_date(DATE_VENTE, 'dd/MM/yyyy')")).alias("date_vente"),
    F.col("BUDGET").alias("budget"),
    F.col("MT_HT").cast("decimal(10,2)").alias("montant_ht"),
    F.col("MT_TTC").cast("decimal(10,2)").alias("montant_ttc"),
    F.col("STATUT").alias("statut"),
    F.col("NUM_BON").alias("numero_bon"),
    F.col("VERSION").alias("version"),
    F.col("CODE_MAG").alias("code_magasin"),
    F.col("CODE_CLIENT").alias("code_client"),
    F.col("CODE_VENDEUR").alias("code_vendeur")
)

# Ajouter l'ID numérique (surrogate key)
window_spec = Window.orderBy("numero_devis")
df_dim_devis = df_temp.select(
    F.row_number().over(window_spec).alias("id_devis"),
    "*"
)

# Sauvegarde avec overwriteSchema pour ajouter la colonne id_devis
df_dim_devis.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"`{CATALOG}`.`{SCHEMA}`.`DimDevis`")
print(f"✓ Table `{CATALOG}`.`{SCHEMA}`.`DimDevis` créée avec succès ({df_dim_devis.count()} lignes)")

print("\nÉchantillon de DimDevis :")
display(df_dim_devis.orderBy(F.desc("date_devis")).limit(10))

##  Création de FactLignesDevis depuis KTP2
Consolidation de tous les fichiers KTP2 (détails des lignes de devis)

In [0]:
# Charger tous les fichiers KTP2
print(" Chargement de tous les fichiers KTP2...")

# Utiliser un wildcard pour charger tous les fichiers CSV dans KTP2/
df_ktp2_all = spark.read.csv(
    BASE_PATH_DEVIS + "KTP2/*.csv",
    sep=";",
    header=True,
    inferSchema=True
)

print(f"✓ Nombre total de lignes de devis chargées : {df_ktp2_all.count()}")
print(f"✓ Nombre de devis distincts : {df_ktp2_all.select('NumDevis').distinct().count()}")

print("\nSchéma des données :")
df_ktp2_all.printSchema()

In [0]:
# Transformation et nettoyage des données pour FactLignesDevis avec jointures vers IDs numériques
print(" Transformation des données avec jointures vers IDs numériques...")

from pyspark.sql.window import Window

df_fact_lignes = df_ktp2_all.withColumn(
    "id_ligne_devis",
    F.monotonically_increasing_id()
)

# Charger toutes les dimensions
print("Chargement des dimensions...")
dim_devis = spark.table("`kitchen-proline-data`.`sales`.`DimDevis`")
dim_magasin = spark.table("`kitchen-proline-data`.`sales`.`DimMagasin`")
dim_client = spark.table("`kitchen-proline-data`.`sales`.`DimClient`")
dim_vendeur = spark.table("`kitchen-proline-data`.`sales`.`DimVendeur`")
dim_pays = spark.table("`kitchen-proline-data`.`sales`.`DimPays`")
dim_produit = spark.table("`kitchen-proline-data`.`sales`.`DimProduit`")
dim_famille = spark.table("`kitchen-proline-data`.`sales`.`DimFamilleProduit`")
dim_marque = spark.table("`kitchen-proline-data`.`sales`.`DimMarque`")
dim_fournisseur = spark.table("`kitchen-proline-data`.`sales`.`DimFournisseur`")
dim_modele = spark.table("`kitchen-proline-data`.`sales`.`DimModele`")
dim_facade = spark.table("`kitchen-proline-data`.`sales`.`DimFacade`")

# Jointure 1: DimDevis (récupère numero_devis + codes pour jointures suivantes)
print(" Jointure avec DimDevis...")
df_with_devis = df_fact_lignes.join(
    dim_devis.select(
        F.col("numero_devis"),
        F.col("code_magasin"),
        F.col("code_client"),
        F.col("code_vendeur")
    ),
    df_fact_lignes["NumDevis"] == dim_devis["numero_devis"],
    "left"
)

# Jointure 2: DimMagasin (récupère IdMagasin + code_pays)
print(" Jointure avec DimMagasin...")
df_with_magasin = df_with_devis.join(
    dim_magasin.select(
        F.col("code_magasin").alias("dim_code_magasin"),
        F.col("id_magasin"),
        F.col("code_pays").alias("magasin_code_pays")
    ),
    df_with_devis["code_magasin"] == F.col("dim_code_magasin"),
    "left"
)

# Jointure 3: DimPays (récupère IdPays)
print("Jointure avec DimPays...")
df_with_pays = df_with_magasin.join(
    dim_pays.select(
        F.col("code_pays").alias("dim_code_pays"),
        F.col("id_pays")
    ),
    df_with_magasin["magasin_code_pays"] == F.col("dim_code_pays"),
    "left"
)

# Jointure 4: DimClient (récupère IdClient)
print("Jointure avec DimClient...")
df_with_client = df_with_pays.join(
    dim_client.select(
        F.col("code_client").alias("dim_code_client"),
        F.col("id_client")
    ),
    df_with_pays["code_client"] == F.col("dim_code_client"),
    "left"
)

# Jointure 5: DimVendeur (récupère IdVendeur)
print("Jointure avec DimVendeur...")
df_with_vendeur = df_with_client.join(
    dim_vendeur.select(
        F.col("code_vendeur").alias("dim_code_vendeur"),
        F.col("id_vendeur")
    ),
    df_with_client["code_vendeur"] == F.col("dim_code_vendeur"),
    "left"
)

# Jointure 6: DimProduit (récupère IdProduit)
# Note: CodeArticle = CodeMarque (3 premiers chars) + CodeProduit (à partir du 4ème char)
print("Jointure avec DimProduit...")
df_with_produit = df_with_vendeur.join(
    dim_produit.select(
        F.col("code_produit").alias("dim_code_produit"),
        F.col("id_produit")
    ),
    F.substring(df_with_vendeur["CodeArticle"], 4, 100) == F.col("dim_code_produit"),
    "left"
)

# Jointure 7: DimFamilleProduit (récupère IdFamille)
print("Jointure avec DimFamilleProduit...")
df_with_famille = df_with_produit.join(
    dim_famille.select(
        F.col("code_famille").alias("dim_code_famille"),
        F.col("id_famille")
    ),
    df_with_produit["CodeFamille"] == F.col("dim_code_famille"),
    "left"
)

# Jointure 8: DimMarque (récupère IdMarque)
print("Jointure avec DimMarque...")
df_with_marque = df_with_famille.join(
    dim_marque.select(
        F.col("code_marque").alias("dim_code_marque"),
        F.col("id_marque")
    ),
    df_with_famille["CodeMarque"] == F.col("dim_code_marque"),
    "left"
)

# Jointure 9: DimFournisseur (récupère IdFournisseur)
print("Jointure avec DimFournisseur...")
df_with_fournisseur = df_with_marque.join(
    dim_fournisseur.select(
        F.col("code_fournisseur").alias("dim_code_fournisseur"),
        F.col("id_fournisseur")
    ),
    df_with_marque["CodeFournisseur"] == F.col("dim_code_fournisseur"),
    "left"
)

# Jointure 10: DimModele (récupère IdModele)
print("Jointure avec DimModele...")
df_with_modele = df_with_fournisseur.join(
    dim_modele.select(
        F.col("code_modele").alias("dim_code_modele"),
        F.col("id_modele")
    ),
    df_with_fournisseur["CODEMODEL"] == F.col("dim_code_modele"),
    "left"
)

# Jointure 11: DimFacade (récupère IdFacade)
print("Jointure avec DimFacade...")
df_with_all_dims = df_with_modele.join(
    dim_facade.select(
        F.col("code_facade").alias("dim_code_facade"),
        F.col("id_facade")
    ),
    df_with_modele["CODEFACADE"].cast("string") == F.col("dim_code_facade"),
    "left"
)

# Sélection finale avec uniquement les IDs numériques
print("Sélection des colonnes finales...")
df_fact_final = df_with_all_dims.select(
    # Clé primaire
    F.col("id_ligne_devis"),
    
    # Clés étrangères numériques vers les dimensions
    F.col("numero_devis").alias("IdDevis"),           # FK vers DimDevis
    F.col("id_magasin").alias("IdMagasin"),          # FK vers DimMagasin
    F.coalesce(F.col("id_client"), F.lit(0)).alias("IdClient"),  # FK vers DimClient (0 = Inconnu)
    F.col("id_vendeur").alias("IdVendeur"),          # FK vers DimVendeur
    F.col("id_pays").alias("IdPays"),                # FK vers DimPays
    F.col("id_produit").alias("IdProduit"),          # FK vers DimProduit
    F.col("id_famille").alias("IdFamille"),          # FK vers DimFamilleProduit
    F.col("id_marque").alias("IdMarque"),            # FK vers DimMarque
    F.col("id_fournisseur").alias("IdFournisseur"),  # FK vers DimFournisseur
    F.col("id_modele").alias("IdModele"),            # FK vers DimModele
    F.col("id_facade").alias("IdFacade"),            # FK vers DimFacade
    
    # Attributs dégénérés
    F.col("Description").alias("description"),
    
    # Mesures (métriques)
    F.col("QTE").alias("quantite"),
    F.col("PV_PUBLIC").cast("decimal(10,2)").alias("prix_public"),
    F.col("PV_TTC").cast("decimal(10,2)").alias("prix_ttc"),
    F.col("PV_HT").cast("decimal(10,2)").alias("prix_ht"),
    F.col("MT_REMISE").cast("decimal(10,2)").alias("montant_remise"),
    F.col("PV_NET_HT").cast("decimal(10,2)").alias("prix_net_ht"),
    F.col("PA_HT").cast("decimal(10,2)").alias("prix_achat_ht"),
    (F.col("PV_NET_HT") - F.col("PA_HT")).cast("decimal(10,2)").alias("marge_brute")
)

# Sauvegarde
print("Sauvegarde de la table de faits...")
df_fact_final.write.mode("overwrite").option("mergeSchema", "true").saveAsTable(f"`{CATALOG}`.`{SCHEMA}`.`FactLignesDevis`")

print("\n FactLignesDevis créée avec toutes les clés étrangères numériques !")
print("\nÉchantillon :")
display(df_fact_final.select(
    "id_ligne_devis",
    "IdDevis",
    "IdMagasin",
    "IdClient",
    "IdProduit",
    "quantite",
    "prix_net_ht",
    "marge_brute"
).limit(10))

## Récapitulatif des tables créées

In [0]:
# Vérification des tables créées
print("="*70)
print("STATISTIQUES DU MODÈLE EN ÉTOILE")
print("="*70)

# DimDevis
dim_devis = spark.table("`kitchen-proline-data`.`sales`.`DimDevis`")
print(f"\n📋 DimDevis")
print(f"   - Nombre de devis : {dim_devis.count()}")
print(f"   - Période : {dim_devis.agg(F.min('date_devis'), F.max('date_devis')).first()}")
print(f"   - Statuts :")
dim_devis.groupBy("statut").count().orderBy(F.desc("count")).show(truncate=False)

# FactLignesDevis
fact_lignes = spark.table("`kitchen-proline-data`.`sales`.`FactLignesDevis`")
print(f"\n FactLignesDevis")
print(f"   - Nombre de lignes : {fact_lignes.count()}")
print(f"   - Montant total HT : {fact_lignes.agg(F.sum('prix_net_ht')).first()[0]:.2f} €")
print(f"   - Marge brute totale : {fact_lignes.agg(F.sum('marge_brute')).first()[0]:.2f} €")

print("\n Clés étrangères numériques dans FactLignesDevis :")
fact_lignes.select(
    "IdDevis",
    "IdMagasin",
    "IdClient",
    "IdVendeur",
    "IdPays",
    "IdProduit",
    "IdFamille",
    "IdMarque",
    "IdFournisseur",
    "IdModele",
    "IdFacade"
).printSchema()

print("\n✓ Vérification des jointures :")
print(f"   - IdDevis NULL : {fact_lignes.filter(F.col('IdDevis').isNull()).count()}")
print(f"   - IdMagasin NULL : {fact_lignes.filter(F.col('IdMagasin').isNull()).count()}")
print(f"   - IdClient NULL : {fact_lignes.filter(F.col('IdClient').isNull()).count()}")
print(f"   - IdProduit NULL : {fact_lignes.filter(F.col('IdProduit').isNull()).count()}")

print("\n="*70)
print("✓ Modèle en étoile complet avec FK numériques !")
print("="*70)
print("\nTables disponibles :")
print("  - 10 Dimensions de références :")
print("    * DimPays, DimMagasin, DimClient, DimVendeur")
print("    * DimProduit, DimFamilleProduit, DimMarque")
print("    * DimFournisseur, DimModele, DimFacade")
print("  - 1 Dimension Devis : DimDevis")
print("  - 1 Table de faits : FactLignesDevis (avec 11 FK numériques)")

In [0]:
from pyspark.sql import functions as F

# Recharger la table actuelle
fact = spark.table("`kitchen-proline-data`.`sales`.`FactLignesDevis`")

# Garder uniquement les bonnes colonnes
fact_clean = fact.select(
    "id_ligne_devis",
    "IdDevis",
    "IdMagasin", 
    "IdClient",
    "IdVendeur",
    "IdPays",
    "IdProduit",
    "IdFamille",
    "IdMarque",
    "IdFournisseur",
    "IdModele",
    "IdFacade",
    "description",
    "quantite",
    "prix_public",
    "prix_ttc",
    "prix_ht",
    "montant_remise",
    "prix_net_ht",
    "prix_achat_ht",
    "marge_brute"
)

# Sauvegarder en REMPLAÇANT complètement la table (pas de mergeSchema)
fact_clean.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("`kitchen-proline-data`.`sales`.`FactLignesDevis`")

# Vérifier
fact_check = spark.table("`kitchen-proline-data`.`sales`.`FactLignesDevis`")
print(f"✓ Nombre de colonnes : {len(fact_check.columns)}")
print(f"✓ Colonnes : {fact_check.columns}")

In [0]:
spark.table("`kitchen-proline-data`.`sales`.`FactLignesDevis`") \
    .select("prix_achat_ht") \
    .filter("prix_achat_ht < 0") \
    .show(10)